In [1]:
import torch

In [45]:
!nvidia-smi

Thu Dec 11 17:51:19 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   61C    P0             29W /   70W |     268MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Device

In [11]:
torch.get_default_dtype()

torch.float32

In [14]:
# torch.set_default_dtype(torch.float64)

In [12]:
a = torch.zeros(4, device="cuda", dtype=torch.float16)

In [13]:
a

tensor([0., 0., 0., 0.], device='cuda:0', dtype=torch.float16)

In [16]:
a + torch.ones(4)

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!

In [18]:
b = a + torch.ones(4, device="cuda")
b.dtype

torch.float32

In [19]:
a.dtype

torch.float16

# Autograd

https://docs.jax.dev/en/latest/advanced-autodiff.html#how-it-s-made-two-foundational-autodiff-functions

In [33]:
x = torch.arange(5.0, requires_grad=True)

In [34]:
y = (x**2).sum()

In [35]:
y

tensor(30., grad_fn=<SumBackward0>)

In [36]:
y.backward()

In [37]:
x.grad

tensor([0., 2., 4., 6., 8.])

In [32]:
x

tensor([0., 1., 2., 3., 4.], requires_grad=True)

# Tensor operations

In [43]:
a = torch.randn(4096, 4096)
b = torch.randn(4096, 4096)

In [44]:
a_cuda = a.to("cuda")
b_cuda = b.to("cuda")

In [65]:
%%time

res = a @ b

CPU times: user 1.66 s, sys: 32.1 ms, total: 1.69 s
Wall time: 1.69 s


In [59]:
%%time

res_cuda = a_cuda @ b_cuda

CPU times: user 0 ns, sys: 787 µs, total: 787 µs
Wall time: 480 µs


In [64]:
%%time

res_cuda = a_cuda @ b_cuda
torch.cuda.synchronize()

CPU times: user 48 ms, sys: 0 ns, total: 48 ms
Wall time: 47.7 ms


# PyTorch lifehacks

## Comments

In [66]:
# comment shapes
def some_func(
    some_input: torch.Tensor,  # (bs, seq_len, emb_dim)
):
    out = some_other_func(some_input)  # (bs, seq_len, n_classes)
    pass

Or you may try https://docs.pytorch.org/docs/stable/named_tensor.html

Caution! Experimental!

## Learn the broadcasting rules!

https://numpy.org/doc/stable/user/basics.broadcasting.html

In [67]:
a = torch.arange(10)
b = torch.arange(-2, 3)
a, b

(tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]), tensor([-2, -1,  0,  1,  2]))

In [68]:
a * b

RuntimeError: The size of tensor a (10) must match the size of tensor b (5) at non-singleton dimension 0

In [69]:
# a: 10 × 1
# b:      5
# c: 10 × 5

c = a.reshape((-1, 1)) * b
c

tensor([[  0,   0,   0,   0,   0],
        [ -2,  -1,   0,   1,   2],
        [ -4,  -2,   0,   2,   4],
        [ -6,  -3,   0,   3,   6],
        [ -8,  -4,   0,   4,   8],
        [-10,  -5,   0,   5,  10],
        [-12,  -6,   0,   6,  12],
        [-14,  -7,   0,   7,  14],
        [-16,  -8,   0,   8,  16],
        [-18,  -9,   0,   9,  18]])

In [70]:
# same
a[:, None]  # (10, 1)
c = a[:, None] * b
c

tensor([[  0,   0,   0,   0,   0],
        [ -2,  -1,   0,   1,   2],
        [ -4,  -2,   0,   2,   4],
        [ -6,  -3,   0,   3,   6],
        [ -8,  -4,   0,   4,   8],
        [-10,  -5,   0,   5,  10],
        [-12,  -6,   0,   6,  12],
        [-14,  -7,   0,   7,  14],
        [-16,  -8,   0,   8,  16],
        [-18,  -9,   0,   9,  18]])

In [71]:
# vice versa
d = a * b[..., None]
d

tensor([[  0,  -2,  -4,  -6,  -8, -10, -12, -14, -16, -18],
        [  0,  -1,  -2,  -3,  -4,  -5,  -6,  -7,  -8,  -9],
        [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
        [  0,   1,   2,   3,   4,   5,   6,   7,   8,   9],
        [  0,   2,   4,   6,   8,  10,  12,  14,  16,  18]])

## Indexing in time series applications

Learn the indexing rules https://numpy.org/doc/stable/user/basics.indexing.html

In [72]:
BS = 3
MAX_LEN = 10
EMB_DIM = 2

In [73]:
x = torch.arange(60).reshape(BS, MAX_LEN, EMB_DIM)  # some embeding of time series in the middle of NN

In [74]:
x

tensor([[[ 0,  1],
         [ 2,  3],
         [ 4,  5],
         [ 6,  7],
         [ 8,  9],
         [10, 11],
         [12, 13],
         [14, 15],
         [16, 17],
         [18, 19]],

        [[20, 21],
         [22, 23],
         [24, 25],
         [26, 27],
         [28, 29],
         [30, 31],
         [32, 33],
         [34, 35],
         [36, 37],
         [38, 39]],

        [[40, 41],
         [42, 43],
         [44, 45],
         [46, 47],
         [48, 49],
         [50, 51],
         [52, 53],
         [54, 55],
         [56, 57],
         [58, 59]]])

Often we have to work with time series of varying length

In [76]:
seq_len = torch.tensor([4, 10, 2])

In [77]:
# WRONG!

# logits = mlp(x[:, -1, :])

How to take the last valid (non-padding) embedding of each element in the batch?

In [81]:
torch.arange(BS), seq_len - 1

(tensor([0, 1, 2]), tensor([3, 9, 1]))

In [80]:
last_emb = x[torch.arange(BS), seq_len - 1]  # (bs, emb)
last_emb

tensor([[ 6,  7],
        [38, 39],
        [42, 43]])

# Reproducibility

FIX ALL SEEDS!!!!!!11

In [101]:
torch.manual_seed(0)

In [102]:
s1 = torch.random.get_rng_state()
s1

tensor([0, 0, 0,  ..., 0, 0, 0], dtype=torch.uint8)

In [104]:
torch.rand((1,))

tensor([0.4963])

In [105]:
s2 = torch.random.get_rng_state()
s2

tensor([0, 0, 0,  ..., 0, 0, 0], dtype=torch.uint8)

In [106]:
(s1 == s2).all()

tensor(False)

1. Fix **all** seeds (all GPUs as well)
2. If checkpointing, save random states to checkpoint

https://yura52.github.io/delu/stable/api/delu.random.html

In [107]:
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)